In [1]:
import torch
import numpy as np
from load_dataset import LoadDataset
from model import MLP
from torch.utils.data import DataLoader
from training import Trainer
from metrics import Evaluator
from gradients import GradMonitor 
import matplotlib.pyplot as plt  
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, random_split


In [2]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Carregar o dataset
data = fetch_california_housing()

# Ver descrição
print(data.DESCR)

# Criar DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # target = preço médio das casas (em $100k)

print(df.head())
print(df.describe())

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [3]:
X = df.drop('target' , axis=1)
import torch

y = torch.tensor(df['target'].values, dtype=torch.float32).unsqueeze(1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
scaler = StandardScaler()

# IMPORTANTE: fit apenas no treino, transform em ambos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ← nunca fit no teste!

# y escalonado
y_mean = y_train.mean(axis=0)
y_std = y_train.std(axis=0)
y_train_scaled = (y_train - y_mean) / y_std
y_test_scaled  = (y_test - y_mean) / y_std
train_data = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), y_train_scaled)
test_data = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), y_test_scaled)

In [5]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [6]:
config_prof = {
    'input_dim': 8,
    'output_dim': 1,   
    'fc': [64,32],
    'criterion':  torch.nn.MSELoss(),
    'optimizer':  torch.optim.Adam,
    'scheduler': {
        'type': torch.optim.lr_scheduler.ReduceLROnPlateau,
        'params': {
            'mode': 'min',
            'factor': 0.5,
            'patience': 10
        }
    },
    'lr': 0.001
}

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
teacher = MLP(config_prof).to(device)
trainer = Trainer(teacher, config_prof)
epochs = 200
ptrain = Evaluator(teacher).mse(train_loader)
patience = 0
for _ in range(epochs):
    print(f'Epoch {_+1}/{epochs} - Train MSE: {ptrain:.4f}', end='\r')
    trainer.train_epoch(train_loader)
    ptrain_old = ptrain
    ptrain = Evaluator(teacher).mse(train_loader)
    if abs(ptrain_old - ptrain) < 1e-3:  # Convergence check
        patience += 1
    else:
        patience = 0
    
    if patience >= 60:  # Early stopping after 5 epochs without improvement
        print(f"Early stopping at epoch {_+1}")
        break

metrics = Evaluator(teacher)
print(metrics.mse(train_loader), metrics.mse(test_loader))
teacher_monitor = GradMonitor(teacher, config_prof)
log_teacher = teacher_monitor.get_gradients(train_loader)


0.14168279517673013 0.20022910683192024


In [8]:
from sklearn.metrics import r2_score

In [9]:
MLP_pred = teacher(torch.tensor(X_test_scaled, dtype=torch.float32).to(device)).cpu().detach().numpy()
r2_score(y_test_scaled, MLP_pred)

0.7957293391227722

In [10]:
print(metrics.mse(train_loader), metrics.mse(test_loader))


0.141682795552147 0.20022910683192024


In [11]:
def get_student_model(T, c, teacher, train_loader, config, max_epochs=20000, patience=100, tol=1e-5):
    student = MLP(config).to(device)
    t_student = Trainer(student, config)
    ptrain = Evaluator(teacher).mse(train_loader)
    sold = ptrain
    patience = 0
    for k in range(max_epochs):
        t_student.train_student(teacher, train_loader, T=T, c=c)
        strain = Evaluator(student).mse(train_loader)
        if (abs(sold - strain)) < tol:  # Convergence check
           patience += 1
        else:
           patience = 0
       
        if patience >= 50:  # Early stopping after 5 epochs without improvement
            break
        sold = strain
    return student, k

##### Será que as informações dentro as redes sao iguais? -> vou dar uma olhada nos gradientes
---

In [14]:
config_student = {
    'input_dim': 8,
    'output_dim': 1,     # RGB
    'fc': [64,32],
    'criterion':  torch.nn.CrossEntropyLoss(),
    'optimizer':  torch.optim.Adam,
    'lr': 0.001
}

In [17]:
student, epochs = get_student_model(T=5, c=0.5, teacher=teacher, train_loader=train_loader, config=config_student)
student_monitor = GradMonitor(student, config_student)
log_student = student_monitor.get_gradients(train_loader) 
metrics_student = Evaluator(student)
print(metrics_student.mse(train_loader), metrics_student.mse(test_loader))

1.0534373741279277 1.0331305350906166


In [16]:
print(metrics_student.mse(train_loader), metrics_student.mse(test_loader))

1.0683571913445642 1.0460851321848788


In [18]:
teacher_info = {'grads' : {}, 'output':[]}
student_info = {'grads' : {}, 'output':[]}
for i in range(len(log_teacher)):
    for key in log_teacher[i]['grads']:
        if key not in teacher_info['grads']:
            teacher_info['grads'][key] = []
        
        teacher_info['grads'][key].append(log_teacher[i]['grads'][key])
        if key in log_student[i]['grads']:
            if key not in student_info['grads']:
                student_info['grads'][key] = []
            student_info['grads'][key].append(log_student[i]['grads'][key])
    teacher_info['output'].append(log_teacher[i]['pred_labels'])
    student_info['output'].append(log_student[i]['pred_labels'])

for key in teacher_info['grads']:
    teacher_info['grads'][key] = torch.cat(teacher_info['grads'][key], dim=0)
    if key in student_info['grads']:
        student_info['grads'][key] = torch.cat(student_info['grads'][key], dim=0)

teacher_info['output'] = torch.cat(teacher_info['output'], dim=0)
student_info['output'] = torch.cat(student_info['output'], dim=0)

In [19]:
idteacher = np.argsort(teacher_info['output'].cpu().numpy())
idstudent = np.argsort(student_info['output'].cpu().numpy())

for key in teacher_info['grads']:
    teacher_info['grads'][key] = teacher_info['grads'][key][idteacher]
    if key in student_info['grads']:
        student_info['grads'][key] = student_info['grads'][key][idstudent]

teacher_info['output'] = teacher_info['output'][idteacher]
student_info['output'] = student_info['output'][idstudent]

In [20]:
from pruning_utils import compute_similarity_matrix

In [ ]:
teacher_sim_matrix = {}
student_sim_matrix = {}
for key in teacher_info['grads']:
    if 'bias' in key:
        continue
    for i in range(teacher_info['grads'][key].shape[1]):
        print(f"Processing {key} with shape {teacher_info['grads'][key][:,i,:].unsqueeze(1).shape}", 'end='\r')
        teacher_sim_matrix[key] = compute_similarity_matrix(teacher_info['grads'][key][:,i,:].unsqueeze(1))
        if key in student_info['grads']:
            student_sim_matrix[key] = compute_similarity_matrix(student_info['grads'][key][:,i,:].unsqueeze(1))

        plot_sim_matrices(teacher_sim_matrix, student_sim_matrix, fname=f"california/sim_matrix_{key}_neuron_{i}", dontshow=True)

Processing layers.fc1.weight with shape torch.Size([16512, 1, 8])


/tmp/ipykernel_78072/2603673813.py:67: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: california/sim_matrix_layers.fc1.weight_neuron_0_layers.fc1.weight.png


In [31]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def plot_sim_matrices(teacher_sim_matrix, student_sim_matrix, fname,dontshow=False):
    """
    Plots similarity matrices for each neuron of each layer,
    comparing teacher (left) vs student (right).

    Args:
        teacher_sim_matrix: dict {layer_key: tensor of shape (N_teacher, nsamples, nsamples)}
        student_sim_matrix: dict {layer_key: tensor of shape (N_student, nsamples, nsamples)}
    """
    for layer_key in teacher_sim_matrix:
        t_mats = teacher_sim_matrix[layer_key]  # (N_t, S, S)
        if layer_key in student_sim_matrix:
            s_mats = student_sim_matrix[layer_key]  # (N_s, S, S)

        # Convert to numpy if tensors
        if hasattr(t_mats, 'cpu'):
            t_mats = t_mats.cpu().numpy()
            if layer_key in student_sim_matrix:
                s_mats = s_mats.cpu().numpy()

        n_teacher = t_mats.shape[0]
        n_student = s_mats.shape[0]
        n_cols    = max(n_teacher, n_student)

        fig = plt.figure(figsize=(3.5 * n_cols, 7))
        fig.suptitle(f"Layer: {layer_key}", fontsize=14, fontweight='bold', y=1.01)

        # Two rows: teacher on top, student on bottom
        outer = gridspec.GridSpec(2, 1, figure=fig, hspace=0.45)

        for row_idx, (label, mats, n_neurons) in enumerate([
            ("Teacher", t_mats, n_teacher),
            ("Student", s_mats, n_student),
        ]):
            inner = gridspec.GridSpecFromSubplotSpec(
                1, n_cols, subplot_spec=outer[row_idx], wspace=0.3
            )

            for neuron_idx in range(n_cols):
                ax = fig.add_subplot(inner[neuron_idx])

                if neuron_idx < n_neurons:
                    mat = mats[neuron_idx]
                    vmin, vmax = mat.min(), mat.max()
                    im = ax.imshow(mat, aspect='auto', cmap='viridis',
                                   vmin=vmin, vmax=vmax, interpolation='nearest')
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                    ax.set_title(f"Neuron {neuron_idx}", fontsize=9)
                    ax.set_xlabel("Sample", fontsize=7)
                    ax.set_ylabel("Sample", fontsize=7)
                    ax.tick_params(labelsize=6)
                else:
                    # Pad empty slots so grids align between teacher/student
                    ax.axis('off')
                    ax.set_facecolor('#f0f0f0')

                # Row label only on the first column
                if neuron_idx == 0:
                    ax.set_ylabel(f"{label}\nSample", fontsize=8, fontweight='bold')



        plt.tight_layout()
        plt.savefig(f"{fname}_{layer_key}.png", dpi=150, bbox_inches='tight')
        print(f"Saved: {fname}_{layer_key}.png")

        if dontshow:
            plt.show()


# ── run ───────────────────────────────────────────────────────────────────────
